In [1]:
import sys
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets openpyxl gradio transformers
print("Libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-l3_z3yir/unsloth_9c6d8913e8c044cb9b1f84f65f1d8d68
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-l3_z3yir/unsloth_9c6d8913e8c044cb9b1f84f65f1d8d68
  Resolved https://github.com/unslothai/unsloth.git to commit 629199e3a67feba9c06bf94c8bd17dc79948a33a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 31.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 70.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Base Model Loading

### Subtask:
Load the TinyLlama-1.1B-Chat model using Unsloth's FastLanguageModel with 4-bit quantization for memory efficiency.


**Reasoning**:
Load the TinyLlama-1.1B-Chat model using the Unsloth library with 4-bit quantization as specified in the instructions to ensure memory efficiency during the fine-tuning process.



In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-chat-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("TinyLlama-1.1B-Chat model and tokenizer loaded successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will load unsloth/tinyllama-chat-bnb-4bit as a legacy tokenizer.


TinyLlama-1.1B-Chat model and tokenizer loaded successfully.


## LoRA Adapter Attachment

### Subtask:
Configure and attach LoRA (Low-Rank Adaptation) adapters to the TinyLlama model to enable efficient fine-tuning.


**Reasoning**:
Incorporate LoRA (Low-Rank Adaptation) adapters into the TinyLlama model using Unsloth's optimized `get_peft_model` method. This configuration uses a rank (r) of 16 and alpha of 32 across all major linear layers to balance trainable parameters and performance, while gradient checkpointing is enabled to significantly reduce VRAM usage.

**Reasoning**:
Configure and attach LoRA (Low-Rank Adaptation) adapters to the TinyLlama model using the Unsloth FastLanguageModel's `get_peft_model` method with the specified parameters to enable memory-efficient fine-tuning.



In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("LoRA adapters successfully attached to the model.")

Unsloth 2026.3.4 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


LoRA adapters successfully attached to the model.


In [7]:
# Cell 4 — Load & prepare all 8 datasets
import pandas as pd
from datasets import Dataset

# ── 1. Load all files (on_bad_lines='skip' handles malformed CSV rows) ─────────
geo_df         = pd.read_excel('/content/Geo-Locations_with_nearbies (1) (2).xlsx')
qa_expert_df   = pd.read_excel('/content/NAL_QA_Dataset (1).xlsx')
market_df      = pd.read_csv('/content/01_annual_market_metrics.csv', on_bad_lines='skip')
news_df        = pd.read_csv('/content/02_news_corpus_articles.csv',  on_bad_lines='skip')
micromarket_df = pd.read_csv('/content/03_micromarket_timeseries.csv', on_bad_lines='skip')
policy_df      = pd.read_csv('/content/04_policy_infrastructure_events.csv', on_bad_lines='skip')
training_qa_df = pd.read_csv('/content/05_training_qa_pairs.csv',     on_bad_lines='skip')
ner_df         = pd.read_csv('/content/06_entity_reference_ner.csv',  on_bad_lines='skip')

print("Files loaded:")
for name, df in [("geo", geo_df), ("qa_expert", qa_expert_df), ("market", market_df),
                 ("news", news_df), ("micromarket", micromarket_df), ("policy", policy_df),
                 ("training_qa", training_qa_df), ("ner", ner_df)]:
    print(f"  {name:15s}: {len(df)} rows")

# ── 2. Helper ──────────────────────────────────────────────────────────────────
combined_data = []

def add(instruction: str, input_text: str, output_text: str) -> None:
    """Append one training example, skipping blank outputs."""
    if not str(output_text).strip() or str(output_text).strip().lower() == 'nan':
        return
    combined_data.append({
        "instruction": instruction.strip(),
        "input":       str(input_text).strip(),
        "output":      str(output_text).strip()
    })

# ── 3. File 1 — NAL QA Dataset  (cols: Question, Response) ────────────────────
# BUG FIX: original code used 'Answer' — correct column is 'Response'
for _, row in qa_expert_df.iterrows():
    add(
        "Answer the following question about the NAL product.",
        str(row.get('Question', '')),
        str(row.get('Response', ''))   # ← was 'Answer' — wrong column name
    )
print(f"After File 1 (NAL QA):        {len(combined_data)} examples")

# ── 4. File 7 — Expert QA pairs  (up-weighted 3×) ─────────────────────────────
for _ in range(3):
    for _, row in training_qa_df.iterrows():
        add(
            f"Provide a detailed answer. [Type: {row.get('question_type','')}, "
            f"Difficulty: {row.get('difficulty','')}, Year: {row.get('year_context','')}]",
            str(row.get('question', '')),
            str(row.get('answer', ''))
        )
print(f"After File 7 (Expert QA ×3):  {len(combined_data)} examples")

# ── 5. File 2 — Geo-locations  (cols: Bangalore_Locations, Bangalore_Zones, ...) ─
# BUG FIX: original code used 'Location' and 'Nearbies' — wrong column names
for _, row in geo_df.iterrows():
    loc   = str(row.get('Bangalore_Locations', ''))
    zone  = str(row.get('Bangalore_Zones', ''))
    ltype = str(row.get('Locality_Areas_Types', ''))
    price = str(row.get('Avg_Property_Price_Per_SqFt', ''))
    c5    = str(row.get('Count_5km', ''))
    c10   = str(row.get('Count_10km', ''))
    add(
        "What zone is this Bangalore location in, and what is the average property price?",
        loc,
        f"{loc} is in the {zone} ({ltype}). "
        f"Average property price: {price} per sq ft. "
        f"Nearby localities: {c5} within 5 km, {c10} within 10 km."
    )
print(f"After File 2 (Geo):            {len(combined_data)} examples")

# ── 6. File 3 — Annual market metrics ─────────────────────────────────────────
for _, row in market_df.iterrows():
    add(
        f"What was the {row.get('metric','')} for {row.get('segment','')} in {row.get('year','')}?",
        "",
        f"The {row.get('metric','')} for {row.get('segment','')} in {row.get('year','')} "
        f"was {row.get('value','')} {row.get('unit','')} (Source: {row.get('source','')}). "
        f"{row.get('notes','')}"
    )
print(f"After File 3 (Market Metrics): {len(combined_data)} examples")

# ── 7. File 4 — News corpus  (cols: headline, body_text, not 'content') ────────
# BUG FIX: original code used 'content' — correct column is 'body_text'
for _, row in news_df.iterrows():
    add(
        f"Summarise the key real estate news from {row.get('micro_market','')} "
        f"in {row.get('year','')} H{row.get('half_year','')}.",
        str(row.get('headline', '')),
        f"{row.get('body_text','')} (Source: {row.get('source_attribution','')})"  # ← was 'content'
    )
print(f"After File 4 (News):           {len(combined_data)} examples")

# ── 8. File 5 — Micromarket time-series ───────────────────────────────────────
for _, row in micromarket_df.iterrows():
    add(
        f"What was the property price trend in {row.get('micro_market','')} in {row.get('year','')}?",
        "",
        f"In {row.get('year','')}, {row.get('micro_market','')} ({row.get('zone','')}) averaged "
        f"INR {row.get('avg_price_psf_inr','')}/sq ft ({row.get('price_range_psf_inr','')}), "
        f"a {row.get('price_yoy_pct','')}% YoY change. "
        f"Key driver: {row.get('key_demand_driver','')}. "
        f"Primary buyers: {row.get('primary_buyer_profile','')}. "
        f"Infrastructure catalyst: {row.get('infrastructure_catalyst','')}."
    )
print(f"After File 5 (Micromarket):    {len(combined_data)} examples")

# ── 9. File 6 — Policy & infrastructure events ────────────────────────────────
for _, row in policy_df.iterrows():
    add(
        f"What was the impact of the following event on the Bengaluru property market?",
        str(row.get('event_name', '')),
        f"{row.get('headline_fact','')}. {row.get('description','')} "
        f"Market effect: {row.get('property_market_effect','')} "
        f"(Micro-markets: {row.get('micro_markets_impacted','')}. Source: {row.get('source','')})"
    )
print(f"After File 6 (Policy):         {len(combined_data)} examples")

# ── 10. File 8 — NER entity reference ─────────────────────────────────────────
for _, row in ner_df.iterrows():
    add(
        "Who is this organisation in the context of Bengaluru real estate?",
        str(row.get('entity_name', '')),
        f"{row.get('entity_name','')} is a {row.get('entity_subtype','')} ({row.get('entity_type','')}) "
        f"active from {row.get('first_active_year','')} to {row.get('last_active_year','')}. "
        f"Role: {row.get('sector_role','')}. "
        f"Primary market: {row.get('micro_market_primary','')}. "
        f"Notable: {row.get('notable_data_point','')}"
    )
print(f"After File 8 (NER):            {len(combined_data)} examples")

# ── 11. Build HuggingFace Dataset & apply chat template ───────────────────────
dataset = Dataset.from_list(combined_data)

def format_chat(sample):
    messages = [
        {"role": "system",    "content": "You are NAL, an expert real estate intelligence assistant for Bengaluru. Answer accurately using your training data."},
        {"role": "user",      "content": f"{sample['instruction']}\n{sample['input']}".strip()},
        {"role": "assistant", "content": sample['output']}
    ]
    sample["text"] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return sample

dataset = dataset.map(format_chat)
dataset = dataset.shuffle(seed=3407)

print(f"\n✅ Final dataset: {len(dataset)} total examples")
print("\nSample record:\n", dataset[0]['text'][:600])

Files loaded:
  geo            : 1680 rows
  qa_expert      : 472 rows
  market         : 84 rows
  news           : 21 rows
  micromarket    : 22 rows
  policy         : 12 rows
  training_qa    : 25 rows
  ner            : 30 rows
After File 1 (NAL QA):        472 examples
After File 7 (Expert QA ×3):  547 examples
After File 2 (Geo):            2227 examples
After File 3 (Market Metrics): 2311 examples
After File 4 (News):           2332 examples
After File 5 (Micromarket):    2354 examples
After File 6 (Policy):         2366 examples
After File 8 (NER):            2396 examples


Map:   0%|          | 0/2396 [00:00<?, ? examples/s]


✅ Final dataset: 2396 total examples

Sample record:
 <|system|>
You are NAL, an expert real estate intelligence assistant for Bengaluru. Answer accurately using your training data.</s>
<|user|>
What zone is this Bangalore location in, and what is the average property price?
Chikkabellandur</s>
<|assistant|>
Chikkabellandur is in the Southeast Zone (Residential Zone). Average property price: ₹12,743.33 per sq ft. Nearby localities: 187 within 5 km, 732 within 10 km.</s>



In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print(f"Training finished. Peak reserved memory: {torch.cuda.max_memory_reserved() / 1024**3:.3f} GB")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2396 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,396 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss
1,2.586418
2,2.540236
3,2.616328
4,2.344928
5,2.105436
6,1.966107
7,1.749562
8,1.379197
9,1.293808
10,1.306712


Training finished. Peak reserved memory: 3.207 GB


**Reasoning**:
The model, tokenizer, and dataset have already been prepared. I will now initialize and run the SFTTrainer with the specified hyperparameters to complete the fine-tuning process as requested in the subtask.



In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# Configure training arguments
training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
)

# Initialize SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = training_args,
)

# Execute training
trainer_stats = trainer.train()

# Print final statistics
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"Training finished. Peak reserved memory: {used_memory} GB")

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/2396 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,396 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss
1,0.417321
2,0.500270
3,0.570621
4,0.370652
5,0.567255
6,0.475573
7,0.670555
8,0.433260
9,0.656119
10,0.825549


Training finished. Peak reserved memory: 3.23 GB


In [11]:
# Save the fine-tuned LoRA adapters and tokenizer
model.save_pretrained("./nal-tinyllama")
tokenizer.save_pretrained("./nal-tinyllama")

print("Fine-tuned LoRA adapters and tokenizer have been successfully saved to './nal-tinyllama'.")

Fine-tuned LoRA adapters and tokenizer have been successfully saved to './nal-tinyllama'.


In [14]:
# 1. Switch model to inference mode
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# 2. Define 3 sample queries based on the Bengaluru dataset
sample_queries = [
    "What zone is Chikkabellandur in and what is the average property price?",
    "Summarise the key real estate news for Whitefield in 2020.",
    "How do I create an account in the NAL app?"
]

# 3. Process each query using the chat template and generate responses
print("--- Inference Smoke Test ---\n")

for query in sample_queries:
    messages = [
        {"role": "system", "content": "You are NAL, an expert real estate intelligence assistant for Bengaluru. Answer accurately using your training data."},
        {"role": "user", "content": query}
    ]

    # Apply chat template and return dict containing 'input_ids'
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    # Explicitly extract the input_ids tensor to get shape and for model input
    input_ids = inputs
    if isinstance(inputs, dict) or hasattr(inputs, 'data'):
        input_ids = inputs.input_ids

    # Generate response using the extracted tensor
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        use_cache=True,
    )

    # Decode and print output (skipping the prompt part by slicing using the tensor's shape)
    response = tokenizer.batch_decode(outputs[:, input_ids.shape[1]:], skip_special_tokens=True)[0]

    print(f"Query: {query}")
    print(f"Response: {response.strip()}\n" + "-"*30 + "\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


--- Inference Smoke Test ---

Query: What zone is Chikkabellandur in and what is the average property price?
Response: Chikkabellandur is in the East Zone (Residential Zone). Average property price: ₹10,033.91 per sq ft. Nearby localities: 215 within 5 km, 730 within 10 km.
------------------------------

Query: Summarise the key real estate news for Whitefield in 2020.
Response: Whitefield recorded a 20 per cent YoY increase in residential sales in 2020, driven by the launch of new projects and the ongoing pandemic-induced demand. The city's top-performing locality, the 10-block residential enclave, recorded a 30 per cent YoY jump in sales in 2020. The pandemic-induced demand led to a 20 per cent YoY increase in new launches in 2020, with 2021 set to witness a 25 per cent YoY jump. The city's top-performing locality, the 10-block residential enclave, recorded a 30 per cent YoY jump in sales in 2020. The pandemic-induced demand led to a 20 per cent YoY increase in new launches in 2020,

# Task
# Plan:
* **Gradio UI Launch**: Initialize and launch the Gradio ChatInterface with streaming capabilities (TextIteratorStreamer + Thread) using the 'Soft Teal' theme. The interface will act as the NAL Real-Estate Intelligence Assistant, providing real-time responses to user queries.
* **Final Task**: Provide a comprehensive project summary, verify the shared public link for the Gradio app, and supply the hyperparameter reference guide for future adjustments.

# Task:
Deploy an interactive streaming Gradio ChatInterface for the fine-tuned NAL Real-Estate Intelligence Assistant. The interface should utilize a 'Soft Teal' theme and leverage `TextIteratorStreamer` with threading to provide real-time, fluid responses to user queries based on the Bengaluru real-estate dataset. After launching, provide a project summary including a verification of the public application link and a hyperparameter reference guide for future model maintenance.

## Gradio UI Launch

### Subtask:
Initialize and launch a Gradio ChatInterface with real-time streaming capabilities and a 'Soft Teal' theme.


**Reasoning**:
I will initialize and launch the Gradio ChatInterface with streaming capabilities and a 'soft' theme as specified in the instructions to complete the real-estate analysis deployment.



In [15]:
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread

def predict(message, history):
    # 1. Format conversation history into chat template
    messages = [{"role": "system", "content": "You are NAL, an expert real estate intelligence assistant for Bengaluru. Answer accurately using your training data."}]
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})
    messages.append({"role": "user", "content": message})

    # 2. Tokenize inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # 3. Setup Streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    # 4. Define generation arguments
    generation_kwargs = dict(
        input_ids=inputs,
        streamer=streamer,
        max_new_tokens=512,
        use_cache=True,
    )

    # 5. Run generation in a separate thread
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # 6. Yield tokens as they are generated for the streaming effect
    partial_text = ""
    for new_text in streamer:
        partial_text += new_text
        yield partial_text

# 7. Launch ChatInterface with 'soft' theme (Teal-like) and public share link
demo = gr.ChatInterface(
    fn=predict,
    title="NAL Real-Estate Intelligence Assistant",
    theme="soft",
    description="Ask me anything about Bengaluru real-estate, market trends, or NAL app features.",
    examples=["What zone is Chikkabellandur in?", "Summarise Whitefield news 2020.", "How to create an account?"]
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8d818d1416d0dddda2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Reasoning**:
The Gradio ChatInterface generated a deprecation warning regarding the message format. I will update the `ChatInterface` to use `type='messages'` and adjust the `predict` function to handle the list-of-dictionaries format for the conversation history. This ensures compatibility with the latest Gradio versions and resolves the warning.



In [20]:
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread

def predict(message, history):
    # 1. Format conversation history into chat template
    messages = [{"role": "system", "content": "You are NAL, an expert real estate intelligence assistant for Bengaluru. Answer accurately using your training data."}]

    # Add existing history to messages
    for msg in history:
        messages.append(msg)

    # Add the latest user message
    messages.append({"role": "user", "content": message})

    # 2. Tokenize inputs
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # 3. Setup Streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    # 4. Define generation arguments with repetition penalty to prevent loops
    generation_kwargs = dict(
        input_ids=inputs,
        streamer=streamer,
        max_new_tokens=512,
        use_cache=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.15, # This prevents the repeating text issue
        do_sample=True
    )

    # 5. Run generation in a separate thread
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # 6. Yield tokens as they are generated for the streaming effect
    partial_text = ""
    for new_text in streamer:
        partial_text += new_text
        yield partial_text

# 7. Launch updated ChatInterface
demo = gr.ChatInterface(
    fn=predict,
    type="messages",
    title="URVI NAL",
    theme="soft",
    description="Ask me anything about Bengaluru real-estate, market trends, or NAL app features. (Fixed repetition logic)",
    examples=["What zone is Chikkabellandur in?", "Summarise Whitefield news 2020.", "How to create an account?"]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d907fb8607a88f4954.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
